# Esperienza 6

In [111]:
import os
import ROOT
import numpy as np
import pandas as pd

ROOT.gStyle.SetOptFit(1111)    # label con chi2/ndf, probabilita e parametri
ROOT.gStyle.SetFitFormat("4.2e")
os.makedirs("Immagini", exist_ok=True)

## Dati 

In [112]:
S = 4040.0          # distanza fotodiodo-fenditura [mm]
err_S = 10.0        # errore su S [mm]
lam_v = 532.0 * 1e-6   # laser verde [mm]
lam_r = 650.0 * 1e-6   #  laser rosso [mm]

## Lettura dati

In [113]:
#x, y, sigma_x, sigma_y
raw = pd.read_excel("Esperienza6c.xlsx", sheet_name="Dati codice", header=None)

verde = raw.iloc[6:67, 4:8].astype("float64").to_numpy()    #  E7:H87
rosso = raw.iloc[6:67, 15:19].astype("float64").to_numpy()   #  J7:M67

# ordino per x crescente
verde = np.ascontiguousarray(verde[np.argsort(verde[:, 0])].T)
rosso = np.ascontiguousarray(rosso[np.argsort(rosso[:, 0])].T)

x_v, y_v, err_x_v, err_y_v = verde
x_r, y_r, err_x_r, err_y_r = rosso

print(rosso)

[[-3.00000000e+01 -2.90000000e+01 -2.80000000e+01 -2.70000000e+01
  -2.60000000e+01 -2.50000000e+01 -2.40000000e+01 -2.30000000e+01
  -2.20000000e+01 -2.10000000e+01 -2.00000000e+01 -1.90000000e+01
  -1.80000000e+01 -1.70000000e+01 -1.60000000e+01 -1.50000000e+01
  -1.40000000e+01 -1.30000000e+01 -1.20000000e+01 -1.10000000e+01
  -1.00000000e+01 -9.00000000e+00 -8.00000000e+00 -7.00000000e+00
  -6.00000000e+00 -5.00000000e+00 -4.00000000e+00 -3.00000000e+00
  -2.00000000e+00 -1.00000000e+00  0.00000000e+00  1.00000000e+00
   2.00000000e+00  3.00000000e+00  4.00000000e+00  5.00000000e+00
   6.00000000e+00  7.00000000e+00  8.00000000e+00  9.00000000e+00
   1.00000000e+01  1.10000000e+01  1.20000000e+01  1.30000000e+01
   1.40000000e+01  1.50000000e+01  1.60000000e+01  1.70000000e+01
   1.80000000e+01  1.90000000e+01  2.00000000e+01  2.10000000e+01
   2.20000000e+01  2.30000000e+01  2.40000000e+01  2.50000000e+01
   2.60000000e+01  2.70000000e+01  2.80000000e+01  2.90000000e+01
   3.00000

## Funzioni

Modello: $y = A\left[\dfrac{\sin(B(x-C))}{B(x-C)}\right]^2 + D$

In [114]:
def sinc2(x, par):
    t = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    D = par[3]
    return A * np.sinc(B * (t - C) / np.pi)**2 + D      # np.sinc(z) = sin(pi z)/(pi z)

def modellopy(t, A, B, C, D):
    return A * np.sinc(B * (t - C) / np.pi)**2 + D

def derivata(x, par):
    t = x
    dt = 1e-6
    return (modellopy(t + dt/2, par[0], par[1], par[2], par[3]) - modellopy(t - dt/2, par[0], par[1], par[2], par[3])) / dt

def retta(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

## Fit sinc² - laser verde

In [115]:
# fit  con solo err_y 
pre_v = ROOT.TF1("pre_v", sinc2, min(x_v), max(x_v), 4)
pre_v.SetParameters(y_v.max(), 1.0, x_v[np.argmax(y_v)], 0)
gr_pre_v = ROOT.TGraphErrors(len(x_v), x_v, y_v, np.zeros_like(x_v), err_y_v)
gr_pre_v.Fit(pre_v, "SQ0")

# fit completo con ROOT
SincFit_v = ROOT.TF1("model_v", sinc2, min(x_v), max(x_v), 4)
SincFit_v.SetParameters(pre_v.GetParameter(0), pre_v.GetParameter(1), pre_v.GetParameter(2), pre_v.GetParameter(3))
SincFit_v.SetParNames("A", "B", "C", "D")

gr1_v = ROOT.TGraphErrors(len(x_v), x_v, y_v, err_x_v, err_y_v)
gr1_v.SetTitle("Dati e fit (verde); x [mm]; V [V]")
gr1_v.SetMarkerStyle(20)
gr1_v.SetMarkerSize(0.7)
gr1_v.SetMarkerColor(ROOT.kAzure + 2)
gr1_v.SetLineColor(ROOT.kAzure + 3)

fit_res_v = gr1_v.Fit(SincFit_v, "S")
print(f"Fit status: {fit_res_v.Status()}")

par_v = [SincFit_v.GetParameter(i) for i in range(4)]
errors_v = [SincFit_v.GetParError(i) for i in range(4)]
chi2_v = fit_res_v.Chi2()
ndf_v = fit_res_v.Ndf()

err_yeff_v = np.sqrt(err_y_v**2 + (derivata(x_v, par_v) * err_x_v)**2)   # errore effettivo

print("Parametri stimati:")
print(f"A = {par_v[0]} ± {errors_v[0]}")
print(f"B = {par_v[1]} ± {errors_v[1]}")
print(f"C = {par_v[2]} ± {errors_v[2]}")
print(f"D = {par_v[3]} ± {errors_v[3]}")
print(f"Chi2 = {chi2_v}, ndf = {ndf_v}")

c1_v = ROOT.TCanvas("c1_v", "Fit (verde)", 800, 600)
c1_v.SetGrid()

gr1_v.Draw("AP")

leg1_v = ROOT.TLegend(0.14, 0.77, 0.42, 0.86)
leg1_v.AddEntry(gr1_v, "Dati", "pe")
leg1_v.AddEntry(SincFit_v, "Fit", "l")
leg1_v.Draw()

c1_v.Draw()
c1_v.SaveAs("Immagini/verde_fit.png")

Fit status: 0****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      15300.1
NDf                       =           57
Edm                       =  1.20191e-06
NCalls                    =          225
A                         =      5.67983   +/-   0.0521492   
B                         =     0.857488   +/-   6.3628e-05  
C                         =    -0.837033   +/-   0.000696431 
D                         =  -0.00269774   +/-   0.000143254 

Parametri stimati:
A = 5.679830920381777 ± 0.05214923245889207
B = 0.8574877156328007 ± 6.36279966473523e-05
C = -0.83703347823165 ± 0.0006964309407038498
D = -0.0026977355367296604 ± 0.00014325390628622107
Chi2 = 15300.09474773809, ndf = 57


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1_v
Info in <TCanvas::Print>: png file Immagini/verde_fit.png has been created


In [116]:
if np.abs(par_v[2]) < 3*errors_v[2]:
    print("C è compatibile con 0")
else:
    print("C NON è compatibile con 0")

print("---")

if np.abs(par_v[3]) < 3*errors_v[3]:
    print("D è compatibile con 0")
else:
    print("D NON è compatibile con 0")

C NON è compatibile con 0
---
D NON è compatibile con 0


## Fit sinc² - laser rosso

In [117]:
# fit preliminare con solo err_y (serve per partire da buoni parametri iniziali)
pre_r = ROOT.TF1("pre_r", sinc2, min(x_r), max(x_r), 4)
pre_r.SetParameters(y_r.max(), 1.0, x_r[np.argmax(y_r)], 0)
gr_pre_r = ROOT.TGraphErrors(len(x_r), x_r, y_r, np.zeros_like(x_r), err_y_r)
gr_pre_r.Fit(pre_r, "SQ0")

# fit completo con ROOT
SincFit_r = ROOT.TF1("model_r", sinc2, min(x_r), max(x_r), 4)
SincFit_r.SetParameters(pre_r.GetParameter(0), pre_r.GetParameter(1), pre_r.GetParameter(2), pre_r.GetParameter(3))
SincFit_r.SetParNames("A", "B", "C", "D")

gr1_r = ROOT.TGraphErrors(len(x_r), x_r, y_r, err_x_r, err_y_r)
gr1_r.SetTitle("Dati e fit (rosso); x [mm]; V [V]")
gr1_r.SetMarkerStyle(20)
gr1_r.SetMarkerSize(0.7)
gr1_r.SetMarkerColor(ROOT.kAzure + 2)
gr1_r.SetLineColor(ROOT.kAzure + 3)

fit_res_r = gr1_r.Fit(SincFit_r, "S")
print(f"Fit status: {fit_res_r.Status()}")

par_r = [SincFit_r.GetParameter(i) for i in range(4)]
errors_r = [SincFit_r.GetParError(i) for i in range(4)]
chi2_r = fit_res_r.Chi2()
ndf_r = fit_res_r.Ndf()

err_yeff_r = np.sqrt(err_y_r**2 + (derivata(x_r, par_r) * err_x_r)**2)   # errore effettivo

print("Parametri stimati:")
print(f"A = {par_r[0]} ± {errors_r[0]}")
print(f"B = {par_r[1]} ± {errors_r[1]}")
print(f"C = {par_r[2]} ± {errors_r[2]}")
print(f"D = {par_r[3]} ± {errors_r[3]}")
print(f"Chi2 = {chi2_r}, ndf = {ndf_r}")

c1_r = ROOT.TCanvas("c1_r", "Fit (rosso)", 800, 600)
c1_r.SetGrid()

gr1_r.Draw("AP")

leg1_r = ROOT.TLegend(0.14, 0.77, 0.42, 0.86)
leg1_r.AddEntry(gr1_r, "Dati", "pe")
leg1_r.AddEntry(SincFit_r, "Fit", "l")
leg1_r.Draw()

c1_r.Draw()
c1_r.SaveAs("Immagini/rosso_fit.png")

Fit status: 0
Parametri stimati:
A = 3.695407253336852 ± 0.04220877229989196
B = 1.0036972892739409 ± 4.527517209872409e-05
C = -0.7561984619163598 ± 0.0004818561760000927
D = -0.001593994233621123 ± 5.8112086831889296e-05
Chi2 = 13026.176183451056, ndf = 57
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      13026.2
NDf                       =           57
Edm                       =  7.99705e-07
NCalls                    =          176
A                         =      3.69541   +/-   0.0422088   
B                         =       1.0037   +/-   4.52752e-05 
C                         =    -0.756198   +/-   0.000481856 
D                         =  -0.00159399   +/-   5.81121e-05 


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1_r
Info in <TCanvas::Print>: png file Immagini/rosso_fit.png has been created


In [118]:
if np.abs(par_r[2]) < 3*errors_r[2]:
    print("C è compatibile con 0")
else:
    print("C NON è compatibile con 0")

print("---")

if np.abs(par_r[3]) < 3*errors_r[3]:
    print("D è compatibile con 0")
else:
    print("D NON è compatibile con 0")

C NON è compatibile con 0
---
D NON è compatibile con 0


## Larghezza della fenditura d

$d = \dfrac{B\lambda S}{2\pi}$, $\quad\sigma_d = \dfrac{\lambda}{2\pi}\sqrt{(S\,\sigma_B)^2 + (B\,\sigma_S)^2}$

In [119]:
d_v = abs(par_v[1]) * lam_v * S / (2 * np.pi)
err_d_v = lam_v / (2 * np.pi) * np.sqrt((S * errors_v[1])**2 + (par_v[1] * err_S)**2)

d_r = abs(par_r[1]) * lam_r * S / (2 * np.pi)
err_d_r = lam_r / (2 * np.pi) * np.sqrt((S * errors_r[1])**2 + (par_r[1] * err_S)**2)

print(f"d verde = {d_v} ± {err_d_v} mm")
print(f"d rosso = {d_r} ± {err_d_r} mm")

d verde = 0.293319567600426 ± 0.0007263646973085922 mm
d rosso = 0.4194861288941505 ± 0.001038504406332301 mm


## Compatibilità tra le due d

$|d_{verde} - d_{rosso}| < 3\sqrt{\sigma_{d,verde}^2 + \sigma_{d,rosso}^2}$

In [120]:
diff_d = np.abs(d_v - d_r)
err_diff_d = np.sqrt(err_d_v**2 + err_d_r**2)

print(f"d verde = {d_v} ± {err_d_v} mm")
print(f"d rosso = {d_r} ± {err_d_r} mm")
print(f"|d verde - d rosso| = {diff_d} mm  ({diff_d / err_diff_d:.1f} sigma)")

if diff_d < 3*err_diff_d:
    print("d verde è compatibile con d rosso")
else:
    print("d verde NON è compatibile con d rosso")

d verde = 0.293319567600426 ± 0.0007263646973085922 mm
d rosso = 0.4194861288941505 ± 0.001038504406332301 mm
|d verde - d rosso| = 0.1261665612937245 mm  (99.6 sigma)
d verde NON è compatibile con d rosso


## Residui normalizzati - laser verde

In [121]:
y_attesi_v = np.array([SincFit_v.Eval(x_val) for x_val in x_v])
residuals_v = y_v - y_attesi_v
residuals_norm_v = residuals_v / err_yeff_v

gaussian_v = ROOT.TF1("gaussian_v", "gaus", min(residuals_norm_v), max(residuals_norm_v), 3)
gaussian_v.SetParNames("const", "mu", "sigma")
gaussian_v.SetLineColor(ROOT.kRed + 1)
gaussian_v.SetLineWidth(3)

hist_v = ROOT.TH1F("hist_v", "Istogramma residui normalizzati (verde)", 10, min(residuals_norm_v), max(residuals_norm_v))
for val in residuals_norm_v:
    hist_v.Fill(val)

gaussian_v.SetParameters(hist_v.GetMaximum(), hist_v.GetMean(), hist_v.GetRMS())

hist_v.SetLineColor(ROOT.kAzure + 3)
hist_v.SetLineWidth(2)
hist_v.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)
hist_v.GetXaxis().SetTitle("residuo normalizzato")
hist_v.GetYaxis().SetTitle("conteggi")

r_v = hist_v.Fit(gaussian_v, "S")
print(f"Fit status: {r_v.Status()}")

par_g_v = [gaussian_v.GetParameter(i) for i in range(3)]
err_g_v = [gaussian_v.GetParError(i) for i in range(3)]

print("Parametri stimati:")
print(f"const = {par_g_v[0]} ± {err_g_v[0]}")
print(f"mu = {par_g_v[1]} ± {err_g_v[1]}")
print(f"sigma = {par_g_v[2]} ± {err_g_v[2]}")

c2_v = ROOT.TCanvas("c2_v", "Istogramma (verde)", 800, 600)
c2_v.SetGrid()

hist_v.Draw()

leg2_v = ROOT.TLegend(0.14, 0.77, 0.42, 0.86)
leg2_v.AddEntry(hist_v, "Residui", "f")
leg2_v.AddEntry(gaussian_v, "Fit gaussiano", "l")
leg2_v.Draw()

c2_v.Draw()
c2_v.SaveAs("Immagini/verde_hist.png")

Warning in <TROOT::Append>: Replacing existing TH1: hist_v (Potential memory leak).


Fit status: 0
Parametri stimati:
const = 23.57176998340997 ± 4.337137277243027
mu = 1.4775830091549573 ± 1.5490952756160332
sigma = 11.337583403983597 ± 1.473418043431594
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      3.37239
NDf                       =            5
Edm                       =  8.51966e-08
NCalls                    =           69
const                     =      23.5718   +/-   4.33714     
mu                        =      1.47758   +/-   1.5491      
sigma                     =      11.3376   +/-   1.47342      	 (limited)


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2_v
Info in <TCanvas::Print>: png file Immagini/verde_hist.png has been created


In [122]:
if np.abs(par_g_v[1]) < 3*err_g_v[1]:
    print("mu è compatibile con 0")
else:
    print("mu NON è compatibile con 0")

print("---")

if np.abs(par_g_v[2] - 1) < 3*err_g_v[2]:
    print("sigma è compatibile con 1")
else:
    print("sigma NON è compatibile con 1")

mu è compatibile con 0
---
sigma NON è compatibile con 1


## Residui normalizzati - laser rosso

In [123]:
y_attesi_r = np.array([SincFit_r.Eval(x_val) for x_val in x_r])
residuals_r = y_r - y_attesi_r
residuals_norm_r = residuals_r / err_yeff_r

gaussian_r = ROOT.TF1("gaussian_r", "gaus", min(residuals_norm_r), max(residuals_norm_r), 3)
gaussian_r.SetParNames("const", "mu", "sigma")
gaussian_r.SetLineColor(ROOT.kRed + 1)
gaussian_r.SetLineWidth(3)

hist_r = ROOT.TH1F("hist_r", "Istogramma residui normalizzati (rosso)", 10, min(residuals_norm_r), max(residuals_norm_r))
for val in residuals_norm_r:
    hist_r.Fill(val)

gaussian_r.SetParameters(hist_r.GetMaximum(), hist_r.GetMean(), hist_r.GetRMS())

hist_r.SetLineColor(ROOT.kAzure + 3)
hist_r.SetLineWidth(2)
hist_r.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)
hist_r.GetXaxis().SetTitle("residuo normalizzato")
hist_r.GetYaxis().SetTitle("conteggi")

r_r = hist_r.Fit(gaussian_r, "S")
print(f"Fit status: {r_r.Status()}")

par_g_r = [gaussian_r.GetParameter(i) for i in range(3)]
err_g_r = [gaussian_r.GetParError(i) for i in range(3)]

print("Parametri stimati:")
print(f"const = {par_g_r[0]} ± {err_g_r[0]}")
print(f"mu = {par_g_r[1]} ± {err_g_r[1]}")
print(f"sigma = {par_g_r[2]} ± {err_g_r[2]}")

c2_r = ROOT.TCanvas("c2_r", "Istogramma (rosso)", 800, 600)
c2_r.SetGrid()

hist_r.Draw()

leg2_r = ROOT.TLegend(0.14, 0.77, 0.42, 0.86)
leg2_r.AddEntry(hist_r, "Residui", "f")
leg2_r.AddEntry(gaussian_r, "Fit gaussiano", "l")
leg2_r.Draw()

c2_r.Draw()
c2_r.SaveAs("Immagini/rosso_hist.png")

Warning in <TROOT::Append>: Replacing existing TH1: hist_r (Potential memory leak).


Fit status: 0
Parametri stimati:
const = 16.092942583511437 ± 3.35486262360963
mu = -2.291960170157422 ± 1.9657003781120537
sigma = 11.780658130478228 ± 2.0494114931570424
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      13.2156
NDf                       =            5
Edm                       =   9.9727e-09
NCalls                    =           73
const                     =      16.0929   +/-   3.35486     
mu                        =     -2.29196   +/-   1.9657      
sigma                     =      11.7807   +/-   2.04941      	 (limited)


Info in <TCanvas::Print>: png file Immagini/rosso_hist.png has been created


In [124]:
if np.abs(par_g_r[1]) < 3*err_g_r[1]:
    print("mu è compatibile con 0")
else:
    print("mu NON è compatibile con 0")

print("---")

if np.abs(par_g_r[2] - 1) < 3*err_g_r[2]:
    print("sigma è compatibile con 1")
else:
    print("sigma NON è compatibile con 1")

mu è compatibile con 0
---
sigma NON è compatibile con 1


## Residui in funzione di x - laser verde

In [125]:
line_v = ROOT.TF1("line_v", retta, min(x_v), max(x_v), 2)
line_v.SetParameters(0, 0)
line_v.SetParNames("m", "q")
line_v.SetLineColor(ROOT.kRed + 1)
line_v.SetLineWidth(2)

gr2_v = ROOT.TGraphErrors(len(x_v), x_v, residuals_v, np.zeros_like(x_v), err_yeff_v)
gr2_v.SetTitle("Residui in funzione di x (verde); x [mm]; V_{misurato} - V_{fit} [V]")
gr2_v.SetMarkerStyle(20)
gr2_v.SetMarkerSize(0.7)
gr2_v.SetMarkerColor(ROOT.kAzure + 2)
gr2_v.SetLineColor(ROOT.kAzure + 3)

fit_res2_v = gr2_v.Fit(line_v, "S")
print(f"Fit status: {fit_res2_v.Status()}")

par_l_v = [line_v.GetParameter(i) for i in range(2)]
err_l_v = [line_v.GetParError(i) for i in range(2)]

print("Parametri stimati:")
print(f"m = {par_l_v[0]} ± {err_l_v[0]}")
print(f"q = {par_l_v[1]} ± {err_l_v[1]}")
print(f"Chi2 = {fit_res2_v.Chi2()}")

if np.abs(par_l_v[0]) < 3*err_l_v[0]:
    print("m è compatibile con 0")
else:
    print("m NON è compatibile con 0")

if np.abs(par_l_v[1]) < 3*err_l_v[1]:
    print("q è compatibile con 0")
else:
    print("q NON è compatibile con 0")

c3_v = ROOT.TCanvas("c3_v", "Residui (verde)", 800, 600)
c3_v.SetGrid()

gr2_v.Draw("AP")

leg3_v = ROOT.TLegend(0.14, 0.77, 0.42, 0.86)
leg3_v.AddEntry(gr2_v, "Residui", "pe")
leg3_v.AddEntry(line_v, "Fit lineare", "l")
leg3_v.Draw()

c3_v.Draw()
c3_v.SaveAs("Immagini/verde_residui.png")

Fit status: 0
Parametri stimati:
m = -3.6597437093426525e-05 ± 3.815758382707833e-06
q = 5.216302923435287e-05 ± 9.475522426601434e-05
Chi2 = 15208.104962217592
m NON è compatibile con 0
q è compatibile con 0
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      15208.1
NDf                       =           59
Edm                       =  1.95817e-20
NCalls                    =           38
m                         = -3.65974e-05   +/-   3.81576e-06 
q                         =   5.2163e-05   +/-   9.47552e-05 


Info in <TCanvas::Print>: png file Immagini/verde_residui.png has been created


## Residui in funzione di x - laser rosso

In [126]:
line_r = ROOT.TF1("line_r", retta, min(x_r), max(x_r), 2)
line_r.SetParameters(0, 0)
line_r.SetParNames("m", "q")
line_r.SetLineColor(ROOT.kRed + 1)
line_r.SetLineWidth(2)

gr2_r = ROOT.TGraphErrors(len(x_r), x_r, residuals_r, np.zeros_like(x_r), err_yeff_r)
gr2_r.SetTitle("Residui in funzione di x (rosso); x [mm]; V_{misurato} - V_{fit} [V]")
gr2_r.SetMarkerStyle(20)
gr2_r.SetMarkerSize(0.7)
gr2_r.SetMarkerColor(ROOT.kAzure + 2)
gr2_r.SetLineColor(ROOT.kAzure + 3)

fit_res2_r = gr2_r.Fit(line_r, "S")
print(f"Fit status: {fit_res2_r.Status()}")

par_l_r = [line_r.GetParameter(i) for i in range(2)]
err_l_r = [line_r.GetParError(i) for i in range(2)]

print("Parametri stimati:")
print(f"m = {par_l_r[0]} ± {err_l_r[0]}")
print(f"q = {par_l_r[1]} ± {err_l_r[1]}")
print(f"Chi2 = {fit_res2_r.Chi2()}")

if np.abs(par_l_r[0]) < 3*err_l_r[0]:
    print("m è compatibile con 0")
else:
    print("m NON è compatibile con 0")

if np.abs(par_l_r[1]) < 3*err_l_r[1]:
    print("q è compatibile con 0")
else:
    print("q NON è compatibile con 0")

c3_r = ROOT.TCanvas("c3_r", "Residui (rosso)", 800, 600)
c3_r.SetGrid()

gr2_r.Draw("AP")

leg3_r = ROOT.TLegend(0.14, 0.77, 0.42, 0.86)
leg3_r.AddEntry(gr2_r, "Residui", "pe")
leg3_r.AddEntry(line_r, "Fit lineare", "l")
leg3_r.Draw()

c3_r.Draw()
c3_r.SaveAs("Immagini/rosso_residui.png")

Fit status: 0
Parametri stimati:
m = -2.0501350484089023e-05 ± 1.8657906458304648e-06
q = 6.776944955717902e-05 ± 4.5533683121514514e-05
Chi2 = 12905.439555449393
m NON è compatibile con 0
q è compatibile con 0
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      12905.4
NDf                       =           59
Edm                       =  1.92816e-21
NCalls                    =           38
m                         = -2.05014e-05   +/-   1.86579e-06 
q                         =  6.77694e-05   +/-   4.55337e-05 


Info in <TCanvas::Print>: png file Immagini/rosso_residui.png has been created


## Shift tra i due laser

Primo minimo è $x_1 = \pi/B = \dfrac{\lambda S}{2d}$, quindi $\dfrac{x_{1,rosso}}{x_{1,verde}} = \dfrac{\lambda_{rosso}}{\lambda_{verde}}$. Lo shift dei minimi deve corrispondere a $\Delta\lambda = \lambda_{rosso} - \lambda_{verde}$.

In [127]:
# posizione del primo minimo rispetto al massimo centrale
x1_v = np.pi / abs(par_v[1])
err_x1_v = np.pi * errors_v[1] / par_v[1]**2
x1_r = np.pi / abs(par_r[1])
err_x1_r = np.pi * errors_r[1] / par_r[1]**2

print(f"primo minimo verde = ±{x1_v} ± {err_x1_v} mm")
print(f"primo minimo rosso = ±{x1_r} ± {err_x1_r} mm")

dx = x1_r - x1_v
err_dx = np.sqrt(err_x1_r**2 + err_x1_v**2)
print(f"shift del primo minimo (rosso - verde) = {dx} ± {err_dx} mm")





# rapporto delle lunghezze d'onda misurato dai fit 
rapporto = x1_r / x1_v
err_rapporto = rapporto * np.sqrt((err_x1_r / x1_r)**2 + (err_x1_v / x1_v)**2)
rapporto_att = lam_r / lam_v
print(f"lambda_rosso / lambda_verde misurato = {rapporto} ± {err_rapporto}")
print(f"lambda_rosso / lambda_verde atteso   = {rapporto_att}")

if np.abs(rapporto - rapporto_att) < 3*err_rapporto:
    print("il rapporto misurato è compatibile con quello atteso")
else:
    print("il rapporto misurato NON è compatibile con quello atteso")

print("---")

# differenza di lunghezza d'onda corrispondente allo shift [nm]
dlam = lam_v * (rapporto - 1) * 1e6
err_dlam = lam_v * err_rapporto * 1e6
dlam_att = (lam_r - lam_v) * 1e6
print(f"Delta lambda dallo shift = {dlam} ± {err_dlam} nm")
print(f"Delta lambda atteso      = {dlam_att} nm")

if np.abs(dlam - dlam_att) < 3*err_dlam:
    print("Delta lambda è compatibile con quello atteso")
else:
    print("Delta lambda NON è compatibile con quello atteso")

primo minimo verde = ±3.6637173878011655 ± 0.0002718581192685972 mm
primo minimo rosso = ±3.13002006398002 ± 0.00014119017614530694 mm
shift del primo minimo (rosso - verde) = -0.5336973238211455 ± 0.0003063356049371369 mm
lambda_rosso / lambda_verde misurato = 0.8543290141324323 ± 7.418815246167042e-05
lambda_rosso / lambda_verde atteso   = 1.2218045112781954
il rapporto misurato NON è compatibile con quello atteso
---
Delta lambda dallo shift = -77.496964481546 ± 0.03946809710960866 nm
Delta lambda atteso      = 117.99999999999994 nm
Delta lambda NON è compatibile con quello atteso


## Grafico con i due laser

Curve centrate ($x - C$) e scalate ($(V-D)$) cosi resta visibile lo spostamento dei minimi e differenza di voltaggio del max

In [128]:
#  x - C , (V - D)
gl_v = ROOT.TGraphErrors(len(x_v), x_v - par_v[2], (y_v - par_v[3]), err_x_v, err_y_v / par_v[0])
gl_v.SetMarkerStyle(20)
gl_v.SetMarkerSize(0.7)
gl_v.SetMarkerColor(ROOT.kGreen + 2)
gl_v.SetLineColor(ROOT.kGreen + 2)

gl_r = ROOT.TGraphErrors(len(x_r), x_r - par_r[2], (y_r - par_r[3]), err_x_r, err_y_r / par_r[0])
gl_r.SetMarkerStyle(20)
gl_r.SetMarkerSize(0.7)
gl_r.SetMarkerColor(ROOT.kRed + 1)
gl_r.SetLineColor(ROOT.kRed + 1)

mg = ROOT.TMultiGraph()
mg.Add(gl_v, "P")
mg.Add(gl_r, "P")
mg.SetTitle("Confronto tra i due laser; x - C [mm]; (V - D) / A")

fl_v = ROOT.TF1("confronto_v", sinc2, min(x_v) - par_v[2], max(x_v) - par_v[2], 4)
fl_v.SetParameters(par_v[0], par_v[1], 0, 0)
fl_v.SetLineColor(ROOT.kGreen + 2)
fl_v.SetLineWidth(2)

fl_r = ROOT.TF1("confronto_r", sinc2, min(x_r) - par_r[2], max(x_r) - par_r[2], 4)
fl_r.SetParameters(par_r[0], par_r[1], 0, 0)
fl_r.SetLineColor(ROOT.kRed + 1)
fl_r.SetLineWidth(2)

# righe tratteggiate sui primi minimi (±x1)
linee = []
for pos, col in [(x1_v, ROOT.kGreen + 2), (x1_r, ROOT.kRed + 1)]:
    for segno in (+1, -1):
        ln = ROOT.TLine(segno * pos, -0.05, segno * pos, 1)
        ln.SetLineColor(col)
        ln.SetLineStyle(2)
        ln.SetLineWidth(2)
        linee.append(ln)

leg4 = ROOT.TLegend(0.60, 0.70, 0.89, 0.89)
leg4.AddEntry(gl_v, "Dati verde", "pe")
leg4.AddEntry(fl_v, "Fit verde", "l")
leg4.AddEntry(gl_r, "Dati rosso", "pe")
leg4.AddEntry(fl_r, "Fit rosso", "l")

c4 = ROOT.TCanvas("c4", "Confronto laser", 900, 650)
c4.SetGrid()
mg.Draw("A")
fl_v.Draw("same")
fl_r.Draw("same")
for ln in linee:
    ln.Draw()
leg4.Draw()


c4.Draw()
c4.SaveAs("Immagini/confronto_fit.png")

Info in <TCanvas::Print>: png file Immagini/confronto_fit.png has been created
